# mini-gpt — Kaggle Olsson-approximate 2-layer experiment

**Goal**: reproduce the induction-head phase change described in Olsson et al. (2022).

Config: n_layers=2, d_model=512, d_k=64, batch=64, 120K steps → 1.97B tokens.

**Usage**: Upload to Kaggle → Settings: GPU T4, Internet ON → Save Version (commit mode).

**Fresh run**: Save Version runs Cell 1–13 in order. Cell 9/10 are resume-only and auto-skip when no checkpoint is attached.

**Resume after crash**: add previous Output as dataset input → set RESUME_PT in Cell 10 → Save Version.

In [ ]:
# Cell 1: GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install dependencies
!pip install tokenizers wandb -q

In [ ]:
# Cell 3: Clone or pull repo
import os
if not os.path.exists('/kaggle/working/mini-gpt'):
    !git clone https://github.com/daisukeabe32/mini-gpt /kaggle/working/mini-gpt -q
    print('cloned')
else:
    !git -C /kaggle/working/mini-gpt pull -q
    print('pulled latest')
%cd /kaggle/working/mini-gpt
!git log --oneline -3

In [ ]:
# Cell 4: W&B login via Kaggle Secret
# Add secret key 'WANDB_API_KEY' in Kaggle notebook Settings > Secrets, then enable 'Attach to notebook'.
import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
import wandb
wandb.login()
print('W&B ready')

In [ ]:
# Cell 4b: Experiment config — the ONLY line to change between datasets
#
# Options:
#   "tinystories"   — TinyStories children's stories (EXP-002, done)
#   "wikitext103"   — Wikipedia articles (EXP-003a)
#   "github_python" — Python source code (EXP-003b)
#
DATASET = "tinystories"

# Derived — do not edit
TOKENIZED_SUBDIR = f"bpe_hf_30000_{DATASET}"
print(f"Dataset : {DATASET}")
print(f"Tokenized dir: data/tokenized/{TOKENIZED_SUBDIR}")


In [ ]:
# Cell 5: Copy tokenized data from Kaggle Dataset -> working dir
import os, shutil

# -- Diagnose what is actually mounted under /kaggle/input --
print('=== /kaggle/input contents ===')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')
print('==============================\n')

FILES = ['tokenizer.json', 'hf_tokenizer.json', 'val_ids.pt', 'train_ids.pt']
DST   = f'data/tokenized/{TOKENIZED_SUBDIR}'

# Search for the matching tokenized directory under /kaggle/input
SRC = None
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == TOKENIZED_SUBDIR and 'train_ids.pt' in files:
        SRC = root
        break

# Fallback: accept any directory that contains train_ids.pt (backward compat)
if SRC is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train_ids.pt' in files:
            SRC = root
            print(f'WARNING: exact match for "{TOKENIZED_SUBDIR}" not found.'
                  f' Using fallback: {root}')
            break

if SRC is None:
    raise FileNotFoundError(
        f'Tokenized data for "{DATASET}" not found under /kaggle/input.\n'
        f'Add the Kaggle Dataset "mini-gpt-{DATASET.replace("_","-")}" via'
        ' right sidebar → Input → + Add Input → Your Datasets.'
    )
print(f'Found dataset at: {SRC}')

os.makedirs(DST, exist_ok=True)
for fname in FILES:
    dst_path = f'{DST}/{fname}'
    if not os.path.exists(dst_path):
        print(f'copying {fname} ...')
        shutil.copy2(f'{SRC}/{fname}', dst_path)
        print(f'  done ({os.path.getsize(dst_path)/1e6:.0f} MB)')
    else:
        print(f'  {fname} — skip (exists)')


In [ ]:
# Cell 6: Verify tokenized data
import torch
train = torch.load(f'data/tokenized/{TOKENIZED_SUBDIR}/train_ids.pt')
val   = torch.load(f'data/tokenized/{TOKENIZED_SUBDIR}/val_ids.pt')
print(f'train: {len(train):,} tokens')
print(f'val  : {len(val):,} tokens')


In [ ]:
# Cell 7: Prepare checkpoint output directory
import os
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')
print('NOTE: in Save Version (commit mode) this directory is persisted as Kaggle Output.')

In [ ]:
# Cell 7b: Resume config — edit ONLY this cell for resume runs
#
# Fresh run : leave RESUME_PT = ''
# Resume    : set RESUME_PT to the path shown by Cell 9
#             e.g. '/kaggle/input/olsson-like/checkpoints/20260512_105419/best.pt'
#
RESUME_PT = ''  # <-- set this for resume, leave empty for fresh run
print(f'RESUME_PT = "{RESUME_PT}"')
print('Mode:', 'RESUME' if RESUME_PT else 'FRESH RUN')

In [ ]:
# Cell 8: START TRAINING — fresh run only (skipped automatically on resume)
import os
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

if RESUME_PT:
    print(f'RESUME_PT is set — skipping fresh training (Cell 10 will resume from {RESUME_PT})')
else:
    # Design intent:
    #   n_layers=2, d_model=512, d_k=64 → Olsson-approximate
    #   batch_size=64, max_iters=120K → 1.97B tokens > phase-change lower bound 1.5B
    #   save_every=2000 → snapshots every 32.8M tokens for emergence curve
    !python -u -m scripts.train_char_gpt \
        --tokenized_dir  data/tokenized/{TOKENIZED_SUBDIR} \
        --block_size  256 \
        --d_model     512 \
        --n_layers      2 \
        --num_heads     8 \
        --d_k          64 \
        --d_ff       2048 \
        --batch_size   64 \
        --max_iters  120000 \
        --lr          3e-4 \
        --min_lr      3e-5 \
        --warmup_iters 1000 \
        --eval_every  1000 \
        --save_every  2000 \
        --checkpoint_dir {CKPT_DIR} \
        2>&1 | tee {CKPT_DIR}/training_log.txt


In [ ]:
# Cell 9: RESUME — find available checkpoints from previous Output
# (safe to run in fresh mode: just prints, does nothing)
import os
print('Available .pt files in /kaggle/input:')
found = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            p = os.path.join(root, f)
            found.append(p)
            print(f'  {p}')
if not found:
    print('  (none — this is a fresh run, no resume needed)')

In [ ]:
# Cell 10: RESUME — runs only when RESUME_PT is set in Cell 7b
import os
CKPT_DIR = '/kaggle/working/checkpoints'

if not RESUME_PT:
    print('RESUME_PT is empty — skipping (fresh run mode)')
elif not os.path.exists(RESUME_PT):
    raise FileNotFoundError(f'RESUME_PT not found: {RESUME_PT}\nCheck Cell 9 output for correct path.')
else:
    print(f'Resuming from: {RESUME_PT}')
    os.makedirs(CKPT_DIR, exist_ok=True)
    !python -u -m scripts.train_char_gpt \
        --tokenized_dir  data/tokenized/{TOKENIZED_SUBDIR} \
        --block_size  256 \
        --d_model     512 \
        --n_layers      2 \
        --num_heads     8 \
        --d_k          64 \
        --d_ff       2048 \
        --batch_size   64 \
        --max_iters  120000 \
        --lr          3e-4 \
        --min_lr      3e-5 \
        --warmup_iters 1000 \
        --eval_every  1000 \
        --save_every  2000 \
        --checkpoint_dir {CKPT_DIR} \
        --resume_from {RESUME_PT} \
        2>&1 | tee {CKPT_DIR}/training_log_resumed.txt


In [ ]:
# Cell 11: Check training progress
import os
CKPT_DIR = '/kaggle/working/checkpoints'
for log_name in ['training_log.txt', 'training_log_resumed.txt']:
    log_path = f'{CKPT_DIR}/{log_name}'
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        lines = [l for l in f if l.startswith('step')]
    print(f'=== {log_name} ({len(lines)} evals) ===')
    if lines:
        print('first :', lines[0].strip())
        print('latest:', lines[-1].strip())

In [ ]:
# Cell 12: Induction head analysis — run after training completes
#
# analyze_attention.py:
#   1. Build repeated random sequence [t0..tN t0..tN] at each seq_len
#   2. Extract per-head attention weights and value outputs
#   3. Compute prefix-matching score (attention diagonal) and
#      copying score (logit-lens: z_h -> W_O_h -> W_U)
#   4. Print summary table across all seq_lens
#   5. Save heatmap grids to figs/attention_grid_seqlenXX.png
#
# NOTE: induction score is distance-dependent — testing multiple seq_lens
# avoids false negatives from a single overly-long sequence.
#
import os, glob
CKPT_DIR = '/kaggle/working/checkpoints'

runs = sorted(glob.glob(f'{CKPT_DIR}/*/best.pt'))
if not runs:
    print('No best.pt found — training may not have finished yet')
else:
    CKPT_PT = runs[-1]
    print(f'Using checkpoint: {CKPT_PT}')
    os.makedirs('figs', exist_ok=True)
    !python -m scripts.analyze_attention \
        --checkpoint {CKPT_PT} \
        --seq_lens 8 16 32 64 \
        --out_dir figs/

In [ ]:
# Cell 13: Emergence curve — induction scores across training snapshots
#
# Scans every step_XXXXX.pt snapshot from this session (/kaggle/working/)
# and from previous sessions added as Input (/kaggle/input/).
# For each snapshot, computes max induction score across ALL heads and
# ALL seq_lens (8, 16, 32, 64). Using multiple seq_lens avoids false negatives
# caused by distance-dependent score decay.
#
import os, glob, torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/kaggle/working/mini-gpt')

from scripts.analyze_attention import load_model, make_repeated_sequence, extract_attention, induction_score

CKPT_DIR  = '/kaggle/working/checkpoints'
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LENS  = [8, 16, 32, 64]   # test all four; take max across seq_lens and heads

snap_paths = sorted(glob.glob(f'{CKPT_DIR}/*/step_*.pt'))
# Also scan previous sessions added as Input datasets
snap_paths += glob.glob('/kaggle/input/**/step_*.pt', recursive=True)
snap_paths = sorted(set(snap_paths))
print(f'Found {len(snap_paths)} snapshots')

results = []   # (tokens, max_score, best_seq_len, best_layer, best_head)
for path in snap_paths:
    try:
        model, tok, cfg = load_model(path, device)
        ckpt   = torch.load(path, map_location=device)
        step   = ckpt['step']
        tokens = step * cfg['batch_size'] * cfg['block_size']

        best_score, best_sl, best_l, best_h = 0.0, 0, 0, 0
        for seq_len in SEQ_LENS:
            max_half = cfg['block_size'] // 2
            sl = min(seq_len, max_half)
            ids, _ = make_repeated_sequence(tok, sl, device)
            all_weights, _ = extract_attention(model, ids)
            for l in range(cfg['n_layers']):
                for h in range(cfg['num_heads']):
                    sc = induction_score(all_weights[l][:, h], sl)
                    if sc > best_score:
                        best_score, best_sl, best_l, best_h = sc, sl, l + 1, h + 1

        results.append((tokens, best_score, best_sl, best_l, best_h))
        print(f'  step={step:>7,} | tokens={tokens/1e9:.3f}B | '
              f'max_induction={best_score:.4f} (L{best_l}H{best_h} @seq_len={best_sl})')
    except Exception as e:
        print(f'  skip {path}: {e}')

if results:
    xs, ys = zip(*[(r[0], r[1]) for r in sorted(results)])
    plt.figure(figsize=(8, 4))
    plt.axhline(0.1, color='red', lw=1, linestyle='--', label='threshold 0.1')
    plt.plot([x/1e9 for x in xs], ys, marker='o', ms=4)
    plt.xlabel('Tokens seen (B)')
    plt.ylabel('Max induction score')
    plt.title('Induction head emergence curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    os.makedirs('figs', exist_ok=True)
    plt.savefig('figs/emergence_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved figs/emergence_curve.png')